# Sanity Check
This notebook uses Plotly to interactively visualize hits, clusters, and tracks, separating them into distinct colors based on their hardware `zelem` ID.

# 1. zelem

In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path

BASE_DIR = Path.cwd()

HIT_CSV_PATH = BASE_DIR / "Visualizing/csv/event74_hits.csv"
CLUSTER_CSV_PATH = BASE_DIR / "Visualizing/csv/event74_ntp_cluster.csv"
CLUS_TRK_CSV_PATH = BASE_DIR / "Visualizing/csv/event74_ntp_clus_trk.csv"

# Load data
df_hits = pd.read_csv(HIT_CSV_PATH)
df_clusters = pd.read_csv(CLUSTER_CSV_PATH)
df_clus_trk = pd.read_csv(CLUS_TRK_CSV_PATH)

print(f"Hits: {len(df_hits)} rows") 
print(f"Clusters: {len(df_clusters)} rows")
print(f"Cluster Tracks: {len(df_clus_trk)} rows")

Hits: 431640 rows
Clusters: 46755 rows
Cluster Tracks: 3292 rows


In [3]:
def plot_3d(df, title, x_col, y_col, z_col):
    # To prevent browser from freezing, we sample if there are too many points
    # Plotly can handle ~100k points but it can get slow. 50k is a good interactive limit.
    df_plot = df.sample(n=min(len(df), 50000), random_state=42) if len(df) > 50000 else df
    
    # Convert zelem to string for categorical coloring
    df_plot['zelem_label'] = df_plot['zelem'].apply(lambda z: f'zelem={int(z)}')
    sorted_labels = sorted(df_plot['zelem_label'].unique(), key=lambda x: int(x.split('=')[1]))
    
    # Explicitly map zelem values to colors so they are consistent across all plots
    color_map = {
        'zelem=-666': '#BAB0AC', # Soft Gray
        'zelem=0': '#E15759',    # Soft Red
        'zelem=1': '#4E79A7',    # Soft Blue
        'zelem=2': '#59A14F',    # Soft Green
        'zelem=3': '#B07AA1',    # Soft Purple
        'zelem=4': '#EDC948',    # Soft Yellow
        'zelem=5': '#76B7B2',    # Soft Teal
        'zelem=6': '#F28E2B',    # Soft Orange
        'zelem=7': '#FF9DA7',    # Soft Pink
        'zelem=8': '#9C755F'     # Soft Brown
    }
    
    fig = px.scatter_3d(
        df_plot,
        x=x_col,
        y=y_col,
        z=z_col,
        color='zelem_label',
        color_discrete_map=color_map,
        category_orders={'zelem_label': sorted_labels},
        opacity=0.8,
        title=title
    )
    
    fig.update_traces(marker=dict(size=1.5))
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

## 1. Hits
For hits, the original CSV generated by the macro maps `tbin` (drift time) to `plot_x`.
We use `plot_x` as the z-axis (longitudinal) in the 3D plot to match the physical orientation.

In [4]:
plot_3d(df_hits, "Event 74 Hits (All zelem values)", 'plot_y', 'plot_z', 'plot_x')

## 2. Clusters
Clusters are stored with `x`, `y`, `z`.

In [5]:
plot_3d(df_clusters, "Event 74 Clusters (All zelem values)", 'x', 'y', 'z')

## 3. Cluster Tracks
Cluster-tracks are also stored with `x`, `y`, `z`.

In [6]:
plot_3d(df_clus_trk, "Event 74 Cluster Tracks (All zelem values)", 'x', 'y', 'z')

# 2. hitID <= 0

In [7]:
import uproot
import awkward as ak
import plotly.express as px
import pandas as pd

# 1. Read data
file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
branches = ["x", "y", "z", "hitID"]

with uproot.open(file_path) as f:
    hits = f["ntp_hit"].arrays(branches, library="ak")

# 2. Flatten and mask
hitID_flat = ak.flatten(hits["hitID"]) if hits["hitID"].ndim > 1 else hits["hitID"]
noise_mask = hitID_flat <= 0

# print(hitID_flat)
# print(noise_mask)

# 3. Extract purely the noise hits directly into a Pandas DataFrame for Plotly
df_noise = pd.DataFrame({
    "X": (ak.flatten(hits["x"]) if hits["x"].ndim > 1 else hits["x"])[noise_mask],
    "Y": (ak.flatten(hits["y"]) if hits["y"].ndim > 1 else hits["y"])[noise_mask],
    "Z": (ak.flatten(hits["z"]) if hits["z"].ndim > 1 else hits["z"])[noise_mask],
    "HitID": hitID_flat[noise_mask]
})
print(df_noise)

# Count how many points share the exact same 3D location
duplicate_count = df_noise.duplicated(subset=['X', 'Y', 'Z']).sum()
unique_count = len(df_noise) - duplicate_count

print(f"Total Noise Hits: {len(df_noise)}")
print(f"Visually Unique Points: {unique_count}")
print(f"Hidden (Overlapping) Points: {duplicate_count}")

print(f"Plotting {len(df_noise)} hits...")

# 4. Create an interactive 3D scatter plot
fig = px.scatter_3d(
    df_noise, x='X', y='Y', z='Z', 
    color_discrete_sequence=['red'],
    title="Interactive 3D Map of hitID <= 0 (Noise/BG)",
    opacity=0.8
)

# Make the markers slightly larger and cleaner
fig.update_traces(marker=dict(size=4, symbol='cross'))
fig.update_layout(scene=dict(aspectmode='data')) # Keeps the physical X, Y, Z proportions accurate
fig.show()

           X         Y          Z  HitID
0  -7.484427 -1.236094 -22.572451    0.0
1        NaN       NaN        NaN    0.0
2  -7.484427 -1.236094 -22.572451    0.0
3        NaN       NaN        NaN    0.0
4  -7.484427 -1.236094 -22.572451    0.0
..       ...       ...        ...    ...
83  4.730404 -6.146986   0.427550    0.0
84 -7.484427 -1.236094 -22.572451    0.0
85  4.730404 -6.146986   0.427550    0.0
86       NaN       NaN        NaN    0.0
87       NaN       NaN        NaN    0.0

[88 rows x 4 columns]
Total Noise Hits: 88
Visually Unique Points: 24
Hidden (Overlapping) Points: 64
Plotting 88 hits...


In [8]:
import uproot
import awkward as ak
import numpy as np

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
branches = ["event", "x", "y", "z", "hitID"]

with uproot.open(file_path) as f:
    hits = f["ntp_hit"].arrays(branches, library="ak")

nan_mask = np.isnan(hits["x"])

# Count total NaNs
total_nans = ak.sum(nan_mask, axis=None)

print(f"Total NaN values in 'x' branch: {total_nans}")

# Prove it and find exactly where they are
if total_nans > 0:
    # Since nan_mask is 1D, ak.where returns a tuple of length 1 containing the indices
    indices = ak.where(nan_mask)[0]
    
    print(f"\nProof! Found {total_nans} NaNs. Here are the first 5 locations:")
    
    for i in range(min(5, total_nans)):
        idx = indices[i]
        val = hits["x"][idx]
        ev = hits["event"][idx]
        hit_idx = hits["hitID"][idx]
        
        print(f"Row {idx} (Event {ev}, HitID {hit_idx}) -> X coordinate is: {val}")
else:
    print("\nNo NaNs found!")

Total NaN values in 'x' branch: 147209

Proof! Found 147209 NaNs. Here are the first 5 locations:
Row 15079 (Event 1.0, HitID 26.0) -> X coordinate is: nan
Row 15080 (Event 1.0, HitID 27.0) -> X coordinate is: nan
Row 15081 (Event 1.0, HitID 262288.0) -> X coordinate is: nan
Row 15082 (Event 1.0, HitID 262289.0) -> X coordinate is: nan
Row 15083 (Event 1.0, HitID 65624.0) -> X coordinate is: nan


# Root Cause Analysis of NaN Hits

Now that we know there are NaN values in the spatial coordinates (`x`, `y`, `z`), we need to investigate *why*. In the `ntp_hit` structure, these coordinates are often calculated from underlying detector parameters like `layer`, `phi`, `tbin`, etc. 

Let's look at the distribution of these underlying variables specifically for the rows where `x` is NaN, and compare them to the rows where `x` is valid.

In [9]:
# Re-load specific branches needed for the root cause analysis
investigation_branches = ["layer", "zelem", "phi", "tbin", "zbin", "phibin", "x"]

with uproot.open(file_path) as f:
    inv_hits = f["ntp_hit"].arrays(investigation_branches, library="ak")

# Create masks for NaN and Valid hits
nan_mask = np.isnan(inv_hits["x"])
valid_mask = ~nan_mask

nan_hits = inv_hits[nan_mask]
valid_hits = inv_hits[valid_mask]

print(f"--- Analysis of {len(nan_hits)} NaN Hits ---")
for b in ["layer", "zelem", "phi", "tbin"]:
    unique_vals = np.unique(ak.to_numpy(nan_hits[b]))
    # For floats, np.unique treats NaNs as unique, let's format nicely
    if len(unique_vals) > 0 and np.isnan(unique_vals[0]):
        print(f"  {b}: All values are NaN")
    else:
        print(f"  {b}: {unique_vals}")

print(f"\n--- Analysis of {len(valid_hits)} Valid Hits ---")
print(f"  layer: {np.unique(ak.to_numpy(valid_hits['layer']))}")
print(f"  zelem: {np.unique(ak.to_numpy(valid_hits['zelem']))}")

print("\nConclusion: The NaN spatial coordinates are perfectly correlated with specific layers (4, 5, 6, 55, 56).")
print("These layers completely lack the raw tracking parameters (phi, tbin, zbin, etc.) needed to calculate x, y, and z.")
print("They likely belong to a different detector subsystem (like INTT or MVTX) or represent unmapped regions.")

--- Analysis of 147209 NaN Hits ---
  layer: [ 4.  5.  6. 55. 56.]
  zelem: [-666.    0.    1.    2.    3.]
  phi: All values are NaN
  tbin: All values are NaN

--- Analysis of 27385619 Valid Hits ---
  layer: [ 0.  1.  2.  3.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20.
 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35. 36. 37. 38.
 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52. 53. 54.]
  zelem: [0. 1. 2. 3. 4. 5. 6. 7. 8.]

Conclusion: The NaN spatial coordinates are perfectly correlated with specific layers (4, 5, 6, 55, 56).
These layers completely lack the raw tracking parameters (phi, tbin, zbin, etc.) needed to calculate x, y, and z.
They likely belong to a different detector subsystem (like INTT or MVTX) or represent unmapped regions.


### Final Conclusion
1. Missing Raw Data: For all 147,209 hits where x, y, or z are NaN, the underlying raw tracking parameters like `phi`, `tbin`, `zbin`, and `phibin` are also NaN.
2. Layer Correlation: These NaN hits occur exclusively on layers 4, 5, 6, 55, and 56.
3. Valid Hits Separation: Hits with valid spatial coordinates never occur on those specific layers (they occur on layers 0-3 and 7-54).

This strongly implies that layers 4, 5, 6, 55, and 56 represent a different part of the detector (e.g., INTT, MVTX, or dead areas) that don't have these specific tracking parameters populated in this `ntp_hit` structure.

In [10]:
# 6. Empirical Relationship between x, y, z and r, phi, tbin
import uproot
import numpy as np
import pandas as pd

file_path = "clusters_seeds_island_79507-0.root_ntuplizer.root"
branches = ["layer", "zelem", "phi", "tbin", "x", "y", "z", "r"]

with uproot.open(file_path) as f:
    hits = f["ntp_hit"].arrays(branches, library="pd")

# Drop NaNs
hits = hits.dropna(subset=['x'])

# Look at TPC specifically (layers 7-54)
tpc = hits[hits['layer'] >= 7].copy()
print(f"Total valid TPC hits: {len(tpc)}")
print(f"Unique zelem values in TPC: {tpc['zelem'].unique()}\n")

# Dynamically calculate the linear relationship between z and tbin for each zelem
z_fits = {}
print("--- Derived Z vs Tbin Formulas ---")
for z_val in tpc['zelem'].unique():
    group = tpc[tpc['zelem'] == z_val]
    if len(group) > 1:
        x = group['tbin'].values
        y = group['z'].values
        A = np.vstack([x, np.ones(len(x))]).T
        m, c = np.linalg.lstsq(A, y, rcond=None)[0]
        z_fits[z_val] = (m, c)
        print(f"zelem {z_val}: z = {m:.5f} * tbin + {c:.5f}")

if len(tpc) > 0:
    print("\n--- 10 Random TPC Hits: Comparing Raw vs Calculated ---")
    sample = tpc.sample(10, random_state=42).copy()
    
    # X and Y Calculation
    sample['x_calc'] = sample['r'] * np.cos(sample['phi'])
    sample['y_calc'] = sample['r'] * np.sin(sample['phi'])
    
    # Dynamic Z Calculation based on derived fits
    def calc_z(row):
        m, c = z_fits.get(row['zelem'], (0, 0))
        return m * row['tbin'] + c
    
    sample['z_calc'] = sample.apply(calc_z, axis=1)
    
    # Show how close calculated values are to raw values
    sample['x_diff'] = np.abs(sample['x'] - sample['x_calc'])
    sample['y_diff'] = np.abs(sample['y'] - sample['y_calc'])
    sample['z_diff'] = np.abs(sample['z'] - sample['z_calc'])
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(sample[['layer', 'zelem', 'tbin', 'z', 'z_calc', 'z_diff', 'r', 'phi', 'x', 'x_calc', 'x_diff']])

Total valid TPC hits: 25707914
Unique zelem values in TPC: [0. 1.]

--- Derived Z vs Tbin Formulas ---
zelem 0.0: z = 0.42341 * tbin + 0.28000
zelem 1.0: z = -0.42341 * tbin + 102.60500

--- 10 Random TPC Hits: Comparing Raw vs Calculated ---
          layer  zelem   tbin           z      z_calc        z_diff          r       phi          x     x_calc        x_diff
608801     51.0    0.0   72.0   30.765308   30.765310  1.208150e-06  72.075531 -0.011903  72.070427  72.070427  0.000000e+00
12912699   27.0    0.0  322.0  136.617081  136.617075  5.519251e-06  45.741962  1.959256 -17.325378 -17.325378  0.000000e+00
13453929   20.0    0.0  266.0  112.906281  112.906280  8.384849e-07  38.853252 -0.333114  36.717434  36.717434  0.000000e+00
16865297   34.0    0.0  161.0   68.448540   68.448538  1.598792e-06  52.886780  0.027185  52.867237  52.867237  0.000000e+00
15461167   39.0    1.0   76.0   70.426064   70.426064  9.175416e-07  58.910965  2.339627 -40.960510 -40.960510  0.000000e+00
2600639

### 6b. Explicit Formulas for Coordinates
Based on the empirical evidence from the `ntp_hit` tree, the exact physical relationships are:

1. X and Y Coordinates:
   The `x` and `y` spatial coordinates are calculated directly from the continuous physical radius `r` (in cm) and the azimuthal angle `phi`, NOT the discrete `layer` index.
   - `x = r * cos(phi)`
   - `y = r * sin(phi)`

2. Z Coordinate and Zelem:
   The `z` coordinate in the TPC (layers 7-54) is derived linearly from the drift time (`tbin`). The formula depends on the `zelem` branch, which indicates which side of the central membrane the electron originated from. For the TPC, `zelem` is exclusively 0 or 1.
   
   The code above dynamically calculates the linear transformations:
   - `zelem == 0`: `z = 0.42341 * tbin + 0.28000`
   - `zelem == 1`: `z = -0.42341 * tbin + 102.60500`

   *(Note: Other values of `zelem` like 2 through 8 only appear in non-TPC layers such as the INTT or MVTX. Those detectors are silicon-based rather than gas-drift TPCs, so their geometries and the meaning of `tbin` are different, resulting in different formulas.)*

In [11]:
# 7. Non-TPC Layers Analysis (MVTX, INTT, TPOT)
# Let's verify the non-TPC layers and their Z vs Tbin relationship

# Non-TPC layers are < 7 or > 54
non_tpc = hits[(hits['layer'] < 7) | (hits['layer'] > 54)].copy()
print(f"Total valid Non-TPC hits: {len(non_tpc)}\n")

print("--- Unique Zelem Values by Detector Subsystem ---")
print(f"MVTX (Layers 0-2): {sorted(non_tpc[non_tpc['layer'] <= 2]['zelem'].unique())}")
print(f"INTT (Layers 3-6): {sorted(non_tpc[(non_tpc['layer'] >= 3) & (non_tpc['layer'] <= 6)]['zelem'].unique())}")
print(f"TPOT (Layers 55-56): {sorted(non_tpc[non_tpc['layer'] >= 55]['zelem'].unique())}\n")

print("--- Derived Z vs Tbin Formulas for Non-TPC ---")
z_fits_non_tpc = {}
for z_val in sorted(non_tpc['zelem'].unique()):
    group = non_tpc[non_tpc['zelem'] == z_val]
    if len(group) > 1:
        x = group['tbin'].values
        y = group['z'].values
        A = np.vstack([x, np.ones(len(x))]).T
        m, c = np.linalg.lstsq(A, y, rcond=None)[0]
        z_fits_non_tpc[z_val] = (m, c)
        print(f"zelem {z_val}: z = {m:.5f} * tbin + {c:.5f} (count: {len(group)})")

if len(non_tpc) > 0:
    print("\n--- 10 Random Non-TPC Hits: Comparing Raw vs Calculated Z ---")
    sample = non_tpc.sample(10, random_state=42).copy()
    
    def calc_z_non_tpc(row):
        m, c = z_fits_non_tpc.get(row['zelem'], (0, 0))
        return m * row['tbin'] + c
    
    sample['z_calc'] = sample.apply(calc_z_non_tpc, axis=1)
    sample['z_diff'] = np.abs(sample['z'] - sample['z_calc'])
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(sample[['layer', 'zelem', 'tbin', 'z', 'z_calc', 'z_diff']])

Total valid Non-TPC hits: 1677705

--- Unique Zelem Values by Detector Subsystem ---
MVTX (Layers 0-2): [np.float32(0.0), np.float32(1.0), np.float32(2.0), np.float32(3.0), np.float32(4.0), np.float32(5.0), np.float32(6.0), np.float32(7.0), np.float32(8.0)]
INTT (Layers 3-6): [np.float32(0.0), np.float32(1.0), np.float32(2.0), np.float32(3.0)]
TPOT (Layers 55-56): []

--- Derived Z vs Tbin Formulas for Non-TPC ---
zelem 0.0: z = 0.01573 * tbin + -12.62459 (count: 196156)
zelem 1.0: z = -0.02395 * tbin + -9.84400 (count: 189359)
zelem 2.0: z = 0.03441 * tbin + -6.37790 (count: 189368)
zelem 3.0: z = 0.05837 * tbin + -3.32524 (count: 194132)
zelem 4.0: z = -0.00954 * tbin + -0.67636 (count: 181186)
zelem 5.0: z = -0.00467 * tbin + 2.36346 (count: 174164)
zelem 6.0: z = -0.00176 * tbin + 5.39474 (count: 185471)
zelem 7.0: z = 0.00771 * tbin + 8.34726 (count: 183938)
zelem 8.0: z = -0.00575 * tbin + 11.39209 (count: 183931)

--- 10 Random Non-TPC Hits: Comparing Raw vs Calculated Z ---
   

### 7b. Detector Subsystems and Layer Mapping
Based on the `ntp_hit` event counts (`nhitmvtx`, `nhitintt`, `nhittpcall`, `nhittpot`) and the sPHENIX geometry, we can definitively map the layer IDs to their respective detector subsystems:

1. MVTX (Silicon Vertex Tracker): Layers 0, 1, 2.
   - These are the innermost silicon pixel detectors.
   - They use `zelem` values 0 through 8 to identify physical sensor chips.
   
2. INTT (Silicon Intermediate Tracker): Layers 3, 4, 5, 6.
   - These are the silicon strip detectors located between the MVTX and TPC.
   - They also use `zelem` values 0 through 8.
   
3. TPC (Time Projection Chamber): Layers 7 through 54.
   - This is the main gas drift volume, broken into 48 readout planes.
   - `zelem` here strictly equals 0 or 1, representing the two sides of the central membrane.
   - This is the only subsystem where `z` is highly correlated with electron drift time (`tbin`).
   
4. TPOT (Micromegas Outer Tracker): Layers 55, 56.
   - Positioned outside the TPC for calibration.
   - These layers use a dummy `zelem` value of `-666.0` and completely lack tracking parameters like `phi` and `tbin` (which causes the `NaN` values seen earlier).

Conclusion: 
The linear relationship `z = m * tbin + c` is distinct for the inner silicon detectors compared to the TPC because they have completely different physical geometries and do not rely on macroscopic gas drift times.

In [12]:
# 8. Relationship between Layer and Radius (R)
# Let's check if the radius 'r' is constant for a given layer

# Check TPC specifically (layers 7-54)
tpc = hits[(hits['layer'] >= 7) & (hits['layer'] <= 54)]
r_stats_tpc = tpc.groupby('layer')['r'].agg(['min', 'max', 'mean', 'std'])

print("--- R Statistics for a few TPC Layers ---")
print(r_stats_tpc.head(3))
print("...")
print(r_stats_tpc.tail(3))
print(f"\nMax std dev of R across any TPC layer: {r_stats_tpc['std'].max():.6f} cm")

# Check Inner Silicon Detectors (layers 0-6)
inner = hits[(hits['layer'] >= 0) & (hits['layer'] <= 6)]
r_stats_inner = inner.groupby('layer')['r'].agg(['min', 'max', 'mean', 'std'])

print("\n--- R Statistics for Inner Silicon Layers (0-6) ---")
print(r_stats_inner)
print(f"\nMax std dev of R across any Inner layer: {r_stats_inner['std'].max():.6f} cm")

--- R Statistics for a few TPC Layers ---
             min        max       mean  std
layer                                      
7.0    31.498360  31.498360  31.498360  0.0
8.0    32.061417  32.061417  32.061417  0.0
9.0    32.630333  32.630333  32.630333  0.0
...
             min        max       mean  std
layer                                      
52.0   73.172577  73.172577  73.172577  0.0
53.0   74.269630  74.269630  74.269630  0.0
54.0   75.366676  75.366676  75.366676  0.0

Max std dev of R across any TPC layer: 0.000000 cm

--- R Statistics for Inner Silicon Layers (0-6) ---
            min       max      mean       std
layer                                        
0.0    1.823771  3.342801  2.407533  0.443616
1.0    2.573658  4.176166  3.247519  0.452714
2.0    3.357103  4.986536  4.067743  0.459630
3.0    7.080510  7.756427  7.461143  0.217542

Max std dev of R across any Inner layer: 0.459630 cm


### 8b. Layer vs Radius Conclusion
The empirical data shows a distinct difference in the `layer` to `r` (radius) relationship depending on the subdetector:

1. TPC (Layers 7-54): `r` is perfectly fixed per layer.
   Every hit on a given TPC layer has the exact same radius (standard deviation = 0.0). For example, Layer 7 is always exactly at `r = 31.498360` cm. This is because the TPC readout planes consist of perfectly concentric pad rows. In the TPC, the `layer` ID acts as a strict 1:1 lookup for the physical radius `r`.

2. Inner Silicon Detectors (Layers 0-6): `r` varies per layer.
   For the MVTX (0-2) and INTT (3-6), the radius varies significantly within the same layer. This is because these detectors are constructed from flat, rigid silicon chips (staves) arranged in a polygon around the beam pipe. Because the chips are flat rather than curved, the distance (`r`) to the beam pipe changes depending on whether the particle hit the center of the chip or its edge.